# Offline Resource Recommendation Label Construction

## Objective

This notebook constructs multi-label recommendation targets for the
context-aware offline hiking resource recommendation component.

The input dataset contains real trail, terrain, emergency-access,
communication-infrastructure, and historical weather context from the
Knuckles study region.

The target represents offline information resources that should be available
to a hiker before entering an area where connectivity may become limited.

The problem is formulated as multi-label classification because multiple
resources may be recommended simultaneously.

Labels are not assigned randomly and no synthetic observations are generated.
Each recommendation rule must be traceable to documented safety reasoning,
observed environmental context, or mandatory safety requirements.

Safety-critical resources are handled conservatively and are not allowed to
depend solely on machine-learning predictions.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

current = Path.cwd().resolve()

PROJECT_ROOT = None

for candidate in [current, *current.parents]:
    if (
        (candidate / "research").is_dir()
        and (candidate / "pubspec.yaml").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from: {current}"
    )

RESEARCH_DIR = PROJECT_ROOT / "research"
DATASET_DIR = RESEARCH_DIR / "dataset"
PROCESSED_DIR = DATASET_DIR / "processed"
RESULTS_DIR = RESEARCH_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_FILE = (
    PROCESSED_DIR
    / "features"
    / "knuckles_trail_context_weather_features.csv"
)

print("Current directory:", current)
print("Project root:", PROJECT_ROOT)
print("Research directory:", RESEARCH_DIR)
print("Feature dataset:", FEATURE_FILE)
print("Exists:", FEATURE_FILE.exists())

Current directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/notebooks
Project root: /Users/mirshad/Desktop/Decentralized-Hiking-AI
Research directory: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research
Feature dataset: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/dataset/processed/features/knuckles_trail_context_weather_features.csv
Exists: True


In [2]:
features = pd.read_csv(
    FEATURE_FILE
)

print("INPUT FEATURE DATASET")
print("=" * 60)

print("Rows:", len(features))
print("Columns:", len(features.columns))

print(
    "Duplicate IDs:",
    features["id"].duplicated().sum()
)

print(
    "Total missing values:",
    features.isna().sum().sum()
)

display(features.head())

INPUT FEATURE DATASET
Rows: 557
Columns: 49
Duplicate IDs: 0
Total missing values: 0


,id,highway,length_km,representative_latitude,representative_longitude,minimum_elevation_m,maximum_elevation_m,elevation_range_m,elevation_gain_m,elevation_loss_m,...,hist_surface_pressure_mean_hpa,hist_precipitation_p95_mm,hist_wind_speed_p95_kmh,hist_wind_gust_p95_kmh,hist_temperature_p05_c,hist_temperature_p95_c,hist_apparent_temperature_p05_c,hist_apparent_temperature_p95_c,hist_rain_hour_fraction,hist_rain_hour_percent
0,way/100146710,footway,1.654853,7.420997,80.712054,1275.355835,1329.643188,54.287354,137.471680,83.390137,...,885.953975,0.9,20.5,60.1,17.5,25.9,18.9,27.5,0.310948,31.094834
1,way/100146711,track,4.966901,7.418119,80.703434,1024.330322,1281.052856,256.722534,485.684814,287.749878,...,885.953975,0.9,20.5,60.1,17.5,25.9,18.9,27.5,0.310948,31.094834
2,way/100148512,track,0.995269,7.429983,80.711814,1332.734375,1420.415405,87.681030,116.777222,29.096191,...,885.953975,0.9,20.5,60.1,17.5,25.9,18.9,27.5,0.310948,31.094834
3,way/100398799,footway,0.073670,7.380225,80.680681,784.558533,789.260803,4.702271,3.243958,4.702271,...,959.606859,1.2,17.3,54.4,20.6,28.2,23.5,32.4,0.342849,34.284867
4,way/100398800,steps,0.195638,7.383670,80.678803,696.232178,753.476257,57.244080,57.244080,0.000000,...,959.606859,1.2,17.3,54.4,20.6,28.2,23.5,32.4,0.342849,34.284867


## Resource Recommendation Architecture

The offline recommendation system uses two complementary layers.

### Mandatory Safety Layer

Essential resources that should always be available for a hiking trip are
included independently of machine-learning predictions:

- Offline map
- General first-aid guidance
- Emergency contact information

These resources are not suitable ML prediction targets because intentionally
assigning them to every hiking context would produce constant labels.

### Context-Aware ML Recommendation Layer

Machine learning is used to recommend additional resources according to trail,
terrain, environmental, emergency-access, and communication-remoteness context.

Candidate contextual resources are:

- Hypothermia guidance
- Heat-illness guidance
- Heavy-rain guidance
- Strong-wind guidance
- Water-safety guidance
- Remote-area emergency guidance

The research problem is formulated as multi-label classification because
multiple contextual resources may be recommended for the same hiking context.

Safety-critical mandatory information is never removed solely because an ML
model fails to recommend it.

In [3]:
MANDATORY_RESOURCES = [
    "offline_map",
    "general_first_aid",
    "emergency_contact_info"
]

PREDICTED_RESOURCES = [
    "hypothermia_guidance",
    "heat_illness_guidance",
    "heavy_rain_guidance",
    "strong_wind_guidance",
    "water_safety_guidance",
    "remote_area_emergency_guidance"
]

print("MANDATORY SAFETY RESOURCES")
print("=" * 60)

for resource in MANDATORY_RESOURCES:
    print("-", resource)

print("\nCONTEXT-AWARE ML TARGETS")
print("=" * 60)

for resource in PREDICTED_RESOURCES:
    print("-", resource)

print(
    "\nNumber of ML target labels:",
    len(PREDICTED_RESOURCES)
)

MANDATORY SAFETY RESOURCES
- offline_map
- general_first_aid
- emergency_contact_info

CONTEXT-AWARE ML TARGETS
- hypothermia_guidance
- heat_illness_guidance
- heavy_rain_guidance
- strong_wind_guidance
- water_safety_guidance
- remote_area_emergency_guidance

Number of ML target labels: 6


In [4]:
candidate_label_features = [
    "maximum_elevation_m",
    "elevation_gain_m",
    "max_slope_deg",

    "nearest_hospital_km",
    "nearest_shelter_km",
    "nearest_drinking_water_km",
    "nearest_ranger_station_km",
    "nearest_communication_structure_km",

    "hist_temperature_min_c",
    "hist_temperature_max_c",
    "hist_temperature_p05_c",
    "hist_temperature_p95_c",

    "hist_apparent_temperature_min_c",
    "hist_apparent_temperature_max_c",
    "hist_apparent_temperature_p05_c",
    "hist_apparent_temperature_p95_c",

    "hist_precipitation_max_mm",
    "hist_precipitation_p95_mm",

    "hist_wind_speed_max_kmh",
    "hist_wind_speed_p95_kmh",
    "hist_wind_gust_max_kmh",
    "hist_wind_gust_p95_kmh",

    "hist_rain_hour_percent"
]

missing_label_features = [
    column
    for column in candidate_label_features
    if column not in features.columns
]

print("LABEL FEATURE CHECK")
print("=" * 60)

print(
    "Missing candidate features:",
    missing_label_features
)

if not missing_label_features:
    display(
        features[
            candidate_label_features
        ].describe().T
    )

LABEL FEATURE CHECK
Missing candidate features: []


,count,mean,std,min,25%,50%,75%,max
maximum_elevation_m,557.0,730.585483,303.589583,79.419571,483.139679,702.879456,953.465454,1856.516113
elevation_gain_m,557.0,48.725940,98.206304,0.000000,4.276978,15.625488,46.969574,1055.033112
max_slope_deg,557.0,21.267620,9.749720,2.659245,13.812297,20.636865,26.975223,60.243735
nearest_hospital_km,557.0,3.063326,2.462704,0.060489,1.325606,2.226207,4.382025,10.871615
nearest_shelter_km,557.0,8.932254,5.447033,0.016000,4.690555,7.759090,11.841401,25.191281
nearest_drinking_water_km,557.0,7.843824,5.360112,0.001456,3.869216,6.272141,10.480470,27.482993
nearest_ranger_station_km,557.0,16.701396,7.064880,0.004376,11.720833,16.647948,22.026100,34.069927
nearest_communication_structure_km,557.0,5.089866,4.102458,0.024421,2.260805,4.241079,7.081592,23.698375
hist_temperature_min_c,557.0,13.710592,1.375196,11.900000,12.500000,15.000000,15.000000,15.200000
hist_temperature_max_c,557.0,33.286715,1.770320,31.100000,31.800000,34.500000,34.500000,37.900000


In [5]:
weather_label_features = [
    "hist_temperature_p05_c",
    "hist_temperature_p95_c",
    "hist_precipitation_p95_mm",
    "hist_wind_speed_p95_kmh",
    "hist_wind_gust_p95_kmh",
    "hist_rain_hour_percent"
]

print("WEATHER FEATURE VARIATION")
print("=" * 60)

for column in weather_label_features:

    print(f"\n{column}")

    print(
        "Unique values:",
        features[column].nunique()
    )

    print(
        sorted(
            features[column]
            .unique()
            .tolist()
        )
    )

WEATHER FEATURE VARIATION

hist_temperature_p05_c
Unique values: 5
[17.5, 18.3, 18.8, 20.6, 22.8]

hist_temperature_p95_c
Unique values: 5
[25.9, 26.4, 27.7, 28.2, 32.8]

hist_precipitation_p95_mm
Unique values: 4
[0.8, 0.9, 1.1, 1.2]

hist_wind_speed_p95_kmh
Unique values: 4
[12.1, 17.3, 17.5, 20.5]

hist_wind_gust_p95_kmh
Unique values: 4
[37.73999999999941, 42.8, 54.4, 60.1]

hist_rain_hour_percent
Unique values: 4
[24.43638189120117, 27.092460752099303, 31.094833880978463, 34.28486673968602]


In [6]:
remoteness_features = [
    "nearest_hospital_km",
    "nearest_shelter_km",
    "nearest_ranger_station_km",
    "nearest_communication_structure_km"
]

display(
    features[
        remoteness_features
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    ).T
)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
nearest_hospital_km,557.0,3.063326,2.462704,0.060489,0.661406,1.325606,2.226207,4.382025,6.580848,8.857937,10.871615
nearest_shelter_km,557.0,8.932254,5.447033,0.016000,3.098345,4.690555,7.759090,11.841401,18.636959,19.405349,25.191281
nearest_ranger_station_km,557.0,16.701396,7.064880,0.004376,6.969752,11.720833,16.647948,22.026100,25.550339,27.733209,34.069927
nearest_communication_structure_km,557.0,5.089866,4.102458,0.024421,1.023804,2.260805,4.241079,7.081592,8.707099,13.216397,23.698375


## Label Construction Methodology

Context-aware recommendation targets are constructed using transparent,
reproducible rules applied to observed and derived trail-context features.

The labels are rule-derived targets rather than direct observations of actual
hiker emergencies or expert decisions. Therefore, model evaluation against
these labels measures the model's ability to reproduce the recommendation
policy.

Historical weather variables represent regional 2021–2025 environmental
profiles. Because only five Knuckles weather reference locations are
available, these variables have limited spatial granularity and are not
interpreted as exact trail-level or real-time weather.

Where dataset-relative thresholds are used, they identify comparatively
higher-risk contexts within the available Knuckles dataset rather than
universal medical or meteorological danger thresholds.

Mandatory resources such as offline maps, general first aid, and emergency
contacts remain outside the ML prediction layer.

In [7]:
terrain_gain_q75 = features[
    "elevation_gain_m"
].quantile(0.75)

terrain_slope_q75 = features[
    "max_slope_deg"
].quantile(0.75)

hospital_q75 = features[
    "nearest_hospital_km"
].quantile(0.75)

shelter_q75 = features[
    "nearest_shelter_km"
].quantile(0.75)

water_q75 = features[
    "nearest_drinking_water_km"
].quantile(0.75)

ranger_q75 = features[
    "nearest_ranger_station_km"
].quantile(0.75)

communication_q75 = features[
    "nearest_communication_structure_km"
].quantile(0.75)

print("DATA-RELATIVE THRESHOLDS")
print("=" * 60)

print(f"Elevation gain Q75: {terrain_gain_q75:.2f} m")
print(f"Maximum slope Q75: {terrain_slope_q75:.2f}°")
print(f"Hospital distance Q75: {hospital_q75:.2f} km")
print(f"Shelter distance Q75: {shelter_q75:.2f} km")
print(f"Drinking water distance Q75: {water_q75:.2f} km")
print(f"Ranger distance Q75: {ranger_q75:.2f} km")
print(f"Communication distance Q75: {communication_q75:.2f} km")

DATA-RELATIVE THRESHOLDS
Elevation gain Q75: 46.97 m
Maximum slope Q75: 26.98°
Hospital distance Q75: 4.38 km
Shelter distance Q75: 11.84 km
Drinking water distance Q75: 10.48 km
Ranger distance Q75: 22.03 km
Communication distance Q75: 7.08 km


In [8]:
labels = features[["id"]].copy()

# Colder historical context + more demanding terrain
labels["hypothermia_guidance"] = (
    (
        features["hist_temperature_p05_c"]
        <= features["hist_temperature_p05_c"].quantile(0.25)
    )
    &
    (
        (features["elevation_gain_m"] >= terrain_gain_q75)
        |
        (features["maximum_elevation_m"]
         >= features["maximum_elevation_m"].quantile(0.75))
    )
).astype(int)

# Comparatively hotter historical context
labels["heat_illness_guidance"] = (
    features["hist_apparent_temperature_p95_c"]
    >= features["hist_apparent_temperature_p95_c"].quantile(0.75)
).astype(int)

# Comparatively wetter historical context
labels["heavy_rain_guidance"] = (
    (
        features["hist_precipitation_p95_mm"]
        >= features["hist_precipitation_p95_mm"].quantile(0.75)
    )
    |
    (
        features["hist_rain_hour_percent"]
        >= features["hist_rain_hour_percent"].quantile(0.75)
    )
).astype(int)

# Comparatively windier historical context
labels["strong_wind_guidance"] = (
    features["hist_wind_gust_p95_kmh"]
    >= features["hist_wind_gust_p95_kmh"].quantile(0.75)
).astype(int)

# Limited proximity to mapped drinking-water support
labels["water_safety_guidance"] = (
    features["nearest_drinking_water_km"]
    >= water_q75
).astype(int)

# Combined emergency/communication remoteness
labels["remote_area_emergency_guidance"] = (
    (
        features["nearest_hospital_km"] >= hospital_q75
    )
    |
    (
        features["nearest_ranger_station_km"] >= ranger_q75
    )
    |
    (
        features["nearest_communication_structure_km"]
        >= communication_q75
    )
).astype(int)

print("Labels created:", len(labels))
display(labels.head())

Labels created: 557


,id,hypothermia_guidance,heat_illness_guidance,heavy_rain_guidance,strong_wind_guidance,water_safety_guidance,remote_area_emergency_guidance
0,way/100146710,1,0,0,1,0,0
1,way/100146711,1,0,0,1,0,0
2,way/100148512,1,0,0,1,0,0
3,way/100398799,0,1,1,0,0,1
4,way/100398800,0,1,1,0,0,1


In [9]:
label_balance = pd.DataFrame({
    "positive": labels[PREDICTED_RESOURCES].sum(),
    "negative": (
        len(labels)
        - labels[PREDICTED_RESOURCES].sum()
    ),
    "positive_percent": (
        labels[PREDICTED_RESOURCES].mean() * 100
    )
})

print("LABEL CLASS BALANCE")
print("=" * 60)

display(label_balance)

LABEL CLASS BALANCE


,positive,negative,positive_percent
hypothermia_guidance,123,434,22.082585
heat_illness_guidance,283,274,50.807899
heavy_rain_guidance,257,300,46.140036
strong_wind_guidance,206,351,36.983842
water_safety_guidance,140,417,25.134650
remote_area_emergency_guidance,247,310,44.344704


In [10]:
labels["contextual_resource_count"] = (
    labels[PREDICTED_RESOURCES].sum(axis=1)
)

print("RESOURCES PER TRAIL")
print("=" * 60)

print(
    labels["contextual_resource_count"]
    .value_counts()
    .sort_index()
)

print("\nSummary:")

print(
    labels["contextual_resource_count"]
    .describe()
)

RESOURCES PER TRAIL
contextual_resource_count
0     42
1     31
2    262
3    187
4     35
Name: count, dtype: int64

Summary:
count    557.000000
mean       2.254937
std        0.938580
min        0.000000
25%        2.000000
50%        2.000000
75%        3.000000
max        4.000000
Name: contextual_resource_count, dtype: float64


In [11]:
print("LABEL VALIDATION")
print("=" * 60)

print("Rows:", len(labels))
print("Duplicate IDs:", labels["id"].duplicated().sum())
print("Missing values:", labels.isna().sum().sum())

invalid_values = 0

for column in PREDICTED_RESOURCES:
    invalid_values += (
        ~labels[column].isin([0, 1])
    ).sum()

print("Invalid binary values:", invalid_values)

LABEL VALIDATION
Rows: 557
Duplicate IDs: 0
Missing values: 0
Invalid binary values: 0


In [12]:
LABEL_DIR = DATASET_DIR / "labels"

LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LABEL_FILE = (
    LABEL_DIR
    / "knuckles_offline_resource_labels.csv"
)

labels.to_csv(
    LABEL_FILE,
    index=False
)

saved_labels = pd.read_csv(
    LABEL_FILE
)

print("LABEL DATASET SAVED")
print("=" * 60)

print("Path:", LABEL_FILE)
print("Shape:", saved_labels.shape)
print(
    "Missing values:",
    saved_labels.isna().sum().sum()
)
print(
    "Duplicate IDs:",
    saved_labels["id"].duplicated().sum()
)

LABEL DATASET SAVED
Path: /Users/mirshad/Desktop/Decentralized-Hiking-AI/research/dataset/labels/knuckles_offline_resource_labels.csv
Shape: (557, 8)
Missing values: 0
Duplicate IDs: 0
